<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
ChromaDB &amp; Indexing
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** ChromaDB ist der Speicher hinter dem Evidence Tool des Meeting- & Research-Briefing-Agent. Metadaten-Filter und Quellenangaben pro Chunk sind kein Nice-to-have, sondern Voraussetzung dafür, dass spätere Antworten überhaupt auf konkrete Protokolle oder Entscheidungsdokumente zurückverweisen können — das ist der Kern der **Handeln**-Fähigkeit im roten Faden des Kurses.

In [3]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M12-ChromaDB-Indexing"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M12_ChromaDB_Indexing",
    "tags": ["m12", "chromadb"],
    "metadata": {"notebook": "M12", "version": "1.0"}
}


✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.4.3
langchain-anthropic                      1.7.5
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.6.6
langchain-google-genai                   4.4.0
langchain-ollama                         1.1.0
langchain-openai                         1.6.7
langchain-protocol                       0.0.19
langchain-text-splitters                 1.1.3
langgraph                                1.2.11
langgraph-checkpoint                     4.2.0
langgraph-checkpoint-sqlite              3.1.1
langgraph-prebuilt                       1.1.0
langgraph-sdk                            0.4.4

IP-Adresse: 34.86.239.142
Hostname: 142.239.86.

In [4]:
#@title 📦 Installationen { display-mode: "form" }
install_packages([
                ('markitdown[all]', 'markitdown'),
                'langchain_huggingface',
                ('unstructured[all-docs]>=0.11.2', 'unstructured'),
                ])

🔄 Installiere markitdown[all]...
✅ markitdown[all] erfolgreich installiert und importiert
🔄 Installiere langchain_huggingface...
✅ langchain_huggingface erfolgreich installiert und importiert
🔄 Installiere unstructured[all-docs]>=0.11.2...
✅ unstructured[all-docs]>=0.11.2 erfolgreich installiert und importiert


In [5]:
#@title 📂 Meeting-Briefing-Korpus laden { display-mode: "form" }
import shutil
from genai_lib.utilities import copy_from_github

KORPUS_QUELLE = "ralf-42/Agenten/02_daten/01_text/korpus_meeting_briefing"
KORPUS_MASKE = "*.pdf"
KORPUS_TARGET = "/content/files"

shutil.rmtree(KORPUS_TARGET, ignore_errors=True)
copy_from_github(
    source=KORPUS_QUELLE,
    target=KORPUS_TARGET,
    mask=KORPUS_MASKE,
)


  Kopiere: 02_daten/01_text/korpus_meeting_briefing/agenda_vorlage_steuerkreis.pdf → /content/files/agenda_vorlage_steuerkreis.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/entscheidungsprotokoll_gesamt.pdf → /content/files/entscheidungsprotokoll_gesamt.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/follow_up_tracker.pdf → /content/files/follow_up_tracker.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/llm_autonomous_agents_survey_2308_11432.pdf → /content/files/llm_autonomous_agents_survey_2308_11432.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/projektauftrag_kompass.pdf → /content/files/projektauftrag_kompass.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/prompt_engineering_survey_2402_07927.pdf → /content/files/prompt_engineering_survey_2402_07927.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/protokoll_fachbereich_workshop_2026-05-05.pdf → /content/files/protokoll_fachbereich_workshop_2026-05-05.pdf
  Kopiere: 02_daten/01_text

['/content/files/agenda_vorlage_steuerkreis.pdf',
 '/content/files/entscheidungsprotokoll_gesamt.pdf',
 '/content/files/follow_up_tracker.pdf',
 '/content/files/llm_autonomous_agents_survey_2308_11432.pdf',
 '/content/files/projektauftrag_kompass.pdf',
 '/content/files/prompt_engineering_survey_2402_07927.pdf',
 '/content/files/protokoll_fachbereich_workshop_2026-05-05.pdf',
 '/content/files/protokoll_kickoff_2026-03-03.pdf',
 '/content/files/protokoll_steuerkreis_2026-03-17.pdf',
 '/content/files/protokoll_steuerkreis_2026-03-31.pdf',
 '/content/files/protokoll_steuerkreis_2026-04-14.pdf',
 '/content/files/protokoll_steuerkreis_2026-04-28.pdf',
 '/content/files/protokoll_steuerkreis_2026-05-19.pdf',
 '/content/files/rag_survey_2312_10997.pdf',
 '/content/files/ragas_evaluation_2309_15217.pdf',
 '/content/files/risikoliste_v1.pdf',
 '/content/files/robust_rag_irrelevant_context_2310_01558.pdf',
 '/content/files/stakeholder_rollenliste.pdf']

# 1 | Übersicht
---

ChromaDB läuft lokal, braucht keine externe Infrastruktur und lässt sich in wenigen Zeilen mit LangChain verbinden. Für den Meeting- & Research-Briefing-Agent ist ChromaDB die erste Vektordatenbank: Mara stellt eine Frage, ChromaDB findet relevante Passagen aus Protokollen, Entscheidungsvorlagen und Fachartikeln.


<p><font color='black' size="5">
Warum ChromaDB?
</font></p>

| Feature | ChromaDB | Nutzen im Kurs |
|---|---|---|
| Lokaler Betrieb | Keine Cloud nötig | Colab- und Laptop-tauglich |
| Persistenz | Dateisystem/SQLite | Index kann wiederverwendet werden |
| Metadaten-Filter | Eingebaut | Suche auf bestimmte PDFs begrenzen |
| LangChain-Integration | Nahtlos | Direkter Übergang zu M13/M14 |


<p><font color='black' size="5">
In diesem Modul
</font></p>

1. ChromaDB einrichten
2. PDF-Dokumente laden und chunken
3. Meeting-Briefing-Korpus indexieren
4. Ähnlichkeitssuche, Score und Metadaten-Filter testen


In [6]:
#@markdown   <p><font size="4" color='green'>  ChromaDB Setup</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph Indexierung
        DOCS["📄 Dokumente"] --> LOADER["📥 Document Loader"]
        LOADER --> SPLITTER["✂️ Text Splitter"]
        SPLITTER --> EMBED["🔢 OpenAI Embeddings"]
        EMBED --> CHROMA[("🗄️ ChromaDB")]
    end
    subgraph Suche
        Q["❓ Query"] --> QE["🔢 Query Embed"]
        QE --> SIM["🔍 Similarity Search"]
        CHROMA --> SIM
        SIM --> RESULT["📋 Top-K Docs"]
    end
'''

mermaid(diagram, width=1100)

# 2 | ChromaDB Setup
---


ChromaDB kann auf zwei Arten betrieben werden: Im **In-Memory-Modus** existieren die Daten nur im RAM — geeignet für schnelle Demos und Tests, aber nach dem Prozessende verloren. Im **persistenten Modus** werden die Daten auf dem Dateisystem gespeichert: lokal direkt im Projektverzeichnis, in Google Colab über ein eingebundenes Google Drive, da das Colab-Dateisystem temporär ist und alle Daten beim Neustart verloren gehen.

**Wichtig für Google Colab:**     
Das Colab-Dateisystem ist temporär — alle Daten gehen beim Neustart verloren.  
Für echte Persistenz muss Google Drive eingebunden werden.


In [7]:
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

CHROMA_DIR = "/content/chroma_m12"

# Embedding-Modell
embeddings = OpenAIEmbeddings(model=EMBEDDINGS)

# ChromaDB persistent initialisieren
vectorstore = Chroma(
    collection_name="meeting_briefing_korpus_m12",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

print(f"\nChromaDB initialisiert")
print(f"  Speicherort: {CHROMA_DIR}")
print(f"  Collection:  meeting_briefing_korpus_m12")
print(f"  Dokumente:   {vectorstore._collection.count()}")


ChromaDB initialisiert
  Speicherort: /content/chroma_m12
  Collection:  meeting_briefing_korpus_m12
  Dokumente:   0


# 3 | Dokumente indexieren
---

M12 legt die Baseline fest: `meeting_briefing_korpus_m12` mit `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`. Die späteren Module dürfen davon abweichen, müssen die Abweichung aber explizit begründen.

📦 **Hinweis zum Modul `genai_lib.briefing_rag`:** Diese Collection bleibt bewusst lokal/ephemer und dient nur der Demonstration dieser Chunking-Variante. Die produktive Kette (Collection `meeting_briefing_korpus_m14`) wird ab M14 auf Google Drive persistiert und steht ab dort als wiederverwendbares Modul `genai_lib.briefing_rag` zur Verfügung — nachfolgende Module (M22, M25, M28, M37) importieren von dort, statt die Kette erneut aufzubauen.



Das Indexieren umfasst drei Schritte:

1. **Laden** – Dokumente einlesen (Text, PDF, Markdown, ...)
2. **Chunken** – In kleinere Abschnitte aufteilen
3. **Einbetten** – Vektoren erstellen und in ChromaDB speichern


<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process_01.png" width="650" alt="Indexierungs-Pipeline: Dokumente → Text Splitter → Embeddings → ChromaDB">
<p><font color='black' size="2">
KI-generiertes Bild
</font></p>

*Indexierungs-Pipeline: Dokumente → Text Splitter → Embeddings → ChromaDB*

<p><font color='black' size="5">
Dokument-Loader im Kurs
</font></p>

Für diesen Meeting-Briefing-Korpus verwenden wir bewusst nur PDFs und `PyPDFLoader`. Das ist robuster als generische Dateilader, weil deren Zusatzabhängigkeiten je nach Paketversion fehleranfällig sein können.

| Loader | Dateityp | Import | Verwendung |
|---|---|---|---|
| `PyPDFLoader` | `.pdf` | `langchain_community.document_loaders` | Standard für `korpus_meeting_briefing/*.pdf` |

Andere Loader für Markdown, Word oder HTML sind nützlich, gehören hier aber nicht zum Kernpfad. Sie werden später bei Transferbeispielen oder eigenen Korpora relevant.


In [8]:
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*triton.*")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_pdf_documents(directory_path: str):
    """Lädt ausschließlich PDFs aus dem Meeting-Briefing-Korpus.

    Bewusst keine generischen Dateilader: Deren Zusatzabhängigkeiten sind je nach
    Pydantic-Version fehleranfällig. PyPDFLoader reicht für den Kurskorpus.
    """
    documents = []
    for pdf_path in sorted(Path(directory_path).glob("*.pdf")):
        loader = PyPDFLoader(str(pdf_path))
        documents.extend(loader.load())
    return documents


# Dokumente laden
dokumente_raw = load_pdf_documents(KORPUS_TARGET)

# Text Splitter: grob genug für Einsteiger, klein genug für präzises Retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(dokumente_raw)

print(f"Geladene PDF-Seiten: {len(dokumente_raw)}")
print(f"Chunks nach Splitting: {len(chunks)}")
for i, chunk in enumerate(chunks[:5]):
    quelle = chunk.metadata.get("source", "?").split("/")[-1]
    print(f"  Chunk {i+1}: {len(chunk.page_content)} Zeichen | Quelle: {quelle}")
if len(chunks) > 5:
    print(f"  ... und {len(chunks) - 5} weitere Chunks")


Geladene PDF-Seiten: 131
Chunks nach Splitting: 1094
  Chunk 1: 430 Zeichen | Quelle: agenda_vorlage_steuerkreis.pdf
  Chunk 2: 438 Zeichen | Quelle: agenda_vorlage_steuerkreis.pdf
  Chunk 3: 468 Zeichen | Quelle: agenda_vorlage_steuerkreis.pdf
  Chunk 4: 419 Zeichen | Quelle: agenda_vorlage_steuerkreis.pdf
  Chunk 5: 448 Zeichen | Quelle: agenda_vorlage_steuerkreis.pdf
  ... und 1089 weitere Chunks


In [9]:
# Chunks in ChromaDB indexieren — nur wenn Collection noch leer ist
BATCH_SIZE = 500  # ChromaDB-Maximum liegt je nach Version bei ca. 5.461; 500 ist robust.

count_vorher = vectorstore._collection.count()
if count_vorher == 0:
    for start in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[start:start + BATCH_SIZE]
        vectorstore.add_documents(batch)
        ende = min(start + BATCH_SIZE, len(chunks))
        print(f"  Batch {start // BATCH_SIZE + 1}: {ende}/{len(chunks)} Chunks indexiert")
    print(f"\nIndexiert: {vectorstore._collection.count()} Dokumente neu angelegt")
else:
    print(f"Collection hat bereits {count_vorher} Einträge — kein erneutes Indexieren")


  Batch 1: 500/1094 Chunks indexiert
  Batch 2: 1000/1094 Chunks indexiert
  Batch 3: 1094/1094 Chunks indexiert

Indexiert: 1094 Dokumente neu angelegt


# 4 | Collections verwalten
---



Eine **Collection** in ChromaDB ist eine benannte Gruppe von Dokumenten – vergleichbar mit einer Tabelle in SQL.



<p><font color='black' size="5">
Wichtige Operationen
</font></p>


| Operation | Methode | Beschreibung |
|-----------|---------|-------------|
| Dokument hinzufügen | `add_documents()` | Neue Chunks indexieren |
| Ähnlichkeitssuche | `similarity_search()` | Top-K ähnliche Dokumente |
| Mit Score | `similarity_search_with_score()` | Suche + Ähnlichkeitswert |
| Metadaten-Filter | `filter={...}` | Nur bestimmte Dokumente |
| Alle Dokumente | `get()` | Rohe Collection-Daten |

In [10]:
from genai_lib.model_config import ROUTER
# Einfache Ähnlichkeitssuche mit Query-Rewrite aus Prompt-Template
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm_rewrite = init_chat_model(ROUTER)
rewrite_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m12_query_rewrite_prompt.md", mode="T")

frage_raw = "Wie unterscheidet sich naive RAG von advanced RAG?"
frage = (rewrite_prompt | llm_rewrite | StrOutputParser()).invoke({"user_question": frage_raw}, config=run_cfg).strip()

ergebnisse = vectorstore.similarity_search(frage, k=2)

print(f"Originalfrage: {frage_raw}")
print(f"Retrieval-Query: {frage}")
print()
print("Top-2 Ergebnisse:")
for i, doc in enumerate(ergebnisse):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"{i+1}. [{quelle}] {doc.page_content[:180]}...")


Originalfrage: Wie unterscheidet sich naive RAG von advanced RAG?
Retrieval-Query: Unterschiede zwischen Naive RAG und Advanced RAG: Definitionen, Architektur, Retrieval- und Generierungsstrategien, Komponenten, Vorteile, Nachteile und Anwendungsfälle

Top-2 Ergebnisse:
1. [rag_survey_2312_10997.pdf] 4
Fig. 3. Comparison between the three paradigms of RAG. (Left) Naive RAG mainly consists of three parts: indexing, retrieval and generation. (Middle)
Advanced RAG proposes multipl...
2. [rag_survey_2312_10997.pdf] specific information. RAG synergistically merges LLMs’ intrin-
sic knowledge with the vast, dynamic repositories of external
databases. This comprehensive review paper offers a det...


In [11]:
# Suche mit Ähnlichkeitsscore
ergebnisse_mit_score = vectorstore.similarity_search_with_score(
    "Warum kann irrelevanter Kontext bei RAG problematisch sein?",
    k=3,
)

mprint("## Suchergebnis mit Scores\n")
rows = []
for doc, score in ergebnisse_mit_score:
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    preview = doc.page_content[:100].replace("\n", " ")
    rows.append(f"| {quelle} | {score:.4f} | {preview}... |")

mprint("| Quelle | Score | Inhalt (Vorschau) |\n|---|---:|---|\n" + "\n".join(rows))


## Suchergebnis mit Scores


| Quelle | Score | Inhalt (Vorschau) |
|---|---:|---|
| rag_survey_2312_10997.pdf | 0.8594 | opportunities for the development of RAG, enabling it to address more complex problems and integrati... |
| rag_survey_2312_10997.pdf | 0.9739 | be worse than no information at all”. Improving RAG’s resistance to such adversarial or counterfactu... |
| rag_survey_2312_10997.pdf | 1.0086 | a robust framework for the systematic evaluation of RAG models, as summarized in Table IV. VII. D IS... |

In [12]:
# Metadaten-Filter: Nur Dokumente aus einer bestimmten PDF-Datei suchen
filter_quelle = "/content/files/protokoll_steuerkreis_2026-04-14.pdf"
gefiltert = vectorstore.similarity_search(
    "Welche Entscheidung wurde zur Vektordatenbank getroffen?",
    k=2,
    filter={"source": filter_quelle},
)

print("Gefilterte Suche:", filter_quelle.split("/")[-1])
for doc in gefiltert:
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"  - {quelle}: {doc.page_content[:140]}...")


Gefilterte Suche: protokoll_steuerkreis_2026-04-14.pdf
  - protokoll_steuerkreis_2026-04-14.pdf: der aktuellen Vektordatenbank-Entscheidung fragt, muss D2 vom 14.04.2026
berücksichtigen. D1 bleibt als historischer Zwischenstand relevant,...
  - protokoll_steuerkreis_2026-04-14.pdf: haben kann.
Entscheidung D2
Der Steuerkreis beschließt, stattdessen eine verwaltete Cloud-Vektordatenbank für den
produktiven Einsatz einzus...


# 5 | ChromaDB Troubleshooting
---

**Häufige Fehler und Lösungen**

| Problem | Symptom | Lösung |
|---------|---------|--------|
| **Doppelte Indexierung** | Dokumente mehrfach vorhanden | IDs prüfen, Collection neu aufbauen |
| **Falsches persist_directory** | Daten weg nach Neustart | Absoluten Pfad verwenden |
| **Falsche Collection** | Leere Suchergebnisse | `collection_name` prüfen |
| **Embedding-Modell-Mismatch** | Dimension Error | Immer gleiches Modell für Index & Query |
| **Zu wenige Chunks** | k > Anzahl Dokumente | k ≤ Anzahl indexierter Chunks setzen |


**Best Practices**

```python
**IMMER: Gleiche Embeddings für Index und Query**
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

**IMMER: Umgebungsabhängigen Pfad verwenden**
import os
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CHROMA_DIR = "/content/drive/MyDrive/chroma_m12"
except ImportError:
    CHROMA_DIR = os.path.abspath("../02_daten/05_sonstiges/chroma_m12")

**IMMER: Große Korpora in Batches indexieren**
BATCH_SIZE = 500
count = vectorstore._collection.count()
if count == 0:
    for start in range(0, len(chunks), BATCH_SIZE):
        vectorstore.add_documents(chunks[start:start + BATCH_SIZE])
    print("Neu indexiert.")
else:
    print(f"Collection hat bereits {count} Einträge — kein erneutes Indexieren nötig.")
```


In [13]:
# Diagnose: Collection-Status prüfen
count = vectorstore._collection.count()
print(f"Dokumente in Collection: {count}")

raw = vectorstore._collection.get(include=["metadatas"])
quellen = sorted(set(m.get("source", "?").split("/")[-1] for m in raw["metadatas"]))
print("Vorhandene Quelldateien:")
for quelle in quellen[:10]:
    print("-", quelle)
if len(quellen) > 10:
    print(f"... und {len(quellen) - 10} weitere")

print("Collection-Diagnose abgeschlossen.")


Dokumente in Collection: 1094
Vorhandene Quelldateien:
- agenda_vorlage_steuerkreis.pdf
- entscheidungsprotokoll_gesamt.pdf
- follow_up_tracker.pdf
- llm_autonomous_agents_survey_2308_11432.pdf
- projektauftrag_kompass.pdf
- prompt_engineering_survey_2402_07927.pdf
- protokoll_fachbereich_workshop_2026-05-05.pdf
- protokoll_kickoff_2026-03-03.pdf
- protokoll_steuerkreis_2026-03-17.pdf
- protokoll_steuerkreis_2026-03-31.pdf
... und 8 weitere
Collection-Diagnose abgeschlossen.


In [14]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M12-ChromaDB-Indexing", limit=3, show_steps=True)

## LangSmith Trace — `M12-ChromaDB-Indexing`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `M12_ChromaDB_Indexing` | ✅ success | 1.8s | 3 |


### Steps — letzter Run: `M12_ChromaDB_Indexing`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `prompt` | `ChatPromptTemplate` | ✅ | 0.0s |
| 2 | `llm` | `ChatOpenAI` | ✅ | 1.8s |
| 3 | `parser` | `StrOutputParser` | ✅ | 0.0s |

# A | Aufgaben
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen, es kann aber auch gerne eine andere Herausforderung angegangen werden.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Wenn bei einer Aufgaben eine Blockade entsteht, kann zum Beispiel Gemini in Google Colab verwendet werden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

<p><font color='black' size="5">
Meeting-Briefing-Korpus indexieren
</font></p>

Baue eine ChromaDB-Wissensdatenbank für die Meeting-/Projekt-PDFs und Fachartikel auf. Der Fokus liegt nicht auf vielen Features, sondern auf einem nachvollziehbaren Index: laden, chunken, suchen, filtern.


**Grundlagen**

1. Erstelle eine ChromaDB-Collection `briefing_demo`.
2. Indexiere mindestens 2 kurze Research-Snippets als `Document`-Objekte.
3. Teste eine Suchanfrage zu RAG-Grundlagen.

**✅ Erledigt wenn:** `collection._collection.count()` gibt mindestens 2 zurück und eine Suche liefert ein Dokument.


In [15]:
# Grundlagen: ChromaDB-Collection + 2 Research-Snippets
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
collection = Chroma(collection_name="briefing_demo", embedding_function=embeddings)

# 1. Mindestens 2 Document-Objekte mit Research-Snippets erstellen
# 2. collection.add_documents(...) aufrufen
# 3. Suchanfrage zu RAG-Grundlagen mit collection.similarity_search() testen

**Aufbau**

1. Nutze den geladenen Meeting-Briefing-Korpus aus `/content/files/*.pdf`.
2. Konfiguriere den `RecursiveCharacterTextSplitter` explizit (`chunk_size=500`, `chunk_overlap=50`).
3. Indexiere die Chunks in einer persistenten ChromaDB-Collection.
4. Führe 3 Suchanfragen mit unterschiedlichen `k`-Werten durch.
5. Zeige mindestens eine Suche mit Metadaten-Filter auf eine konkrete PDF-Datei.
6. Stelle die Ergebnisse als Markdown-Tabelle dar.

**✅ Erledigt wenn:** Die Collection enthält PDF-Chunks und drei Suchanfragen mit unterschiedlichen `k`-Werten liefern nachvollziehbare Ergebnisse.


In [16]:
# Aufbau: Meeting-Briefing-Korpus + Chunking + k-Vergleich
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. PDFs aus KORPUS_TARGET laden (PyPDFLoader)
# 2. RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50) anwenden
# 3. Chunks in eine persistente ChromaDB-Collection (briefing_collection) indexieren
# 4. Drei Suchanfragen mit unterschiedlichen k-Werten durchfuehren (results)
# 5. Ergebnisse als Markdown-Tabelle darstellen

**Vertiefung**

- Nutze Metadaten-Filter, um die Suche auf eine bestimmte PDF-Quelle einzuschränken.
- Vergleiche Treffer mit und ohne Filter.
- Aktiviere LangSmith-Tracing für die Suchanfragen und analysiere den Trace.

**✅ Erledigt wenn:** Der Metadaten-Filter schränkt Ergebnisse korrekt ein und LangSmith ist aktiv.


In [17]:
# Vertiefung: Metadaten-Filter auf PDF-Quelle
# Startpunkt: briefing_collection aus Aufbau

# 1. Eine konkrete PDF-Quelle (quelle) waehlen
# 2. Suche mit filter={"source": quelle} durchfuehren und mit ungefiltert vergleichen
# 3. LangSmith-Tracing ist bereits aktiv (siehe Setup-Zelle) - Trace prüfen

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [RAG-Pipeline](https://editor.p5js.org/ralf.bendig.rb/full/RrfB3nCwK)
- [Embeddings 2D](https://editor.p5js.org/ralf.bendig.rb/full/LPjLkzWbE)
- [Embeddings 3D](https://editor.p5js.org/ralf.bendig.rb/full/gFBwB2E8n)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Einsteiger ChromaDB](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-chromadb.html)
- [Embeddings](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/embeddings.html)
- [Tokenizing & Chunking](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/tokenizing-chunking.html)
- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)
- [Context Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/context-engineering.html)
